In [41]:
import csv, random
from datetime import date, timedelta
from typing import NamedTuple
from faker import Faker

SEED = 26
random.seed(SEED)
fake = Faker("en_IE")
Faker.seed(SEED)

# Fixed date limit
TODAY = date(2026, 11, 15)

class Coverage(NamedTuple):
    limit: int # max payout per claim (€)
    deductible: int # paid by the customer per claim (€)

class Peril(NamedTuple):
    annual_frequency: float # annual premium rate (% of sum insured)
    loss_min: int # min payout per claim (€)
    loss_max: int # max payout per claim (€)

In [42]:
def random_customer(i):
    customer_since = fake.date_between(date(2024, 1, 1), TODAY)
    return {
        "customer_id": f"CUST-{i:05d}",
        "full_name": f"{fake.first_name()} {fake.last_name()}",
        "county": fake.county(),
        "customer_since": customer_since,
    }

In [43]:
PRODUCTS: dict[str, dict[str, Coverage]] = {
    "HOME-BASIC": {
        "fire": Coverage(limit=150_000, deductible=500),
        "theft": Coverage(limit=10_000, deductible=250),
        "storm": Coverage(limit=50_000, deductible=500)
    },
    "HOME-STD": {
        "fire": Coverage(limit=300_000, deductible=350),
        "theft": Coverage(limit=25_000, deductible=250),
        "storm": Coverage(limit=100_000, deductible=350),
        "water_damage": Coverage(limit=30_000, deductible=350)
    },
    "HOME-PLUS": {
        "fire": Coverage(limit=500_000, deductible=250),
        "theft": Coverage(limit=50_000, deductible=150),
        "storm": Coverage(limit=200_000, deductible=250),
        "water_damage": Coverage(limit=60_000, deductible=250),
        "accidental_damage": Coverage(limit=15_000, deductible=150)
    },
}

RATES = {
    "HOME-BASIC": (0.0015, 0.0025),   # ~€225 – €375
    "HOME-STD":   (0.0013, 0.0020),   # ~€390 – €600
    "HOME-PLUS":  (0.0012, 0.0018),   # ~€600 – €900
}

def random_policy(i, customer):
    product = random.choice(list(PRODUCTS))

    since = customer["customer_since"]
    status = random.choices(["active", "lapsed", "cancelled"], [85, 10, 5])[0]

    sum_insured = PRODUCTS[product]["fire"].limit
    annual_premium = round(sum_insured * random.uniform(*RATES[product]), 2)

    if status == "lapsed" and since > TODAY - timedelta(days=366):
        status = "active"

    if status == "active":
        start = fake.date_between(max(since, TODAY - timedelta(days=364)), TODAY)
    elif status == "lapsed":
        start = fake.date_between(since, TODAY - timedelta(days=366))
    else:
        start = fake.date_between(since, TODAY)

    end = start + timedelta(days=365)
    cancelled_at = (fake.date_between(start, min(end, TODAY)) if status == "cancelled" else None)

    return {
        "policy_id": f"POL-{start.year}-{i:05d}",
        "customer_id": customer["customer_id"],
        "product_code": product,
        "status": status,
        "start_date": start,
        "end_date": end,
        "cancelled_at": cancelled_at,
        "annual_premium": annual_premium,
        "sum_insured": sum_insured,
    }

In [44]:
def build_coverages(policy):
    return [
        {
            "policy_id": policy["policy_id"],
            "peril": peril,
            "limit_amount": coverage.limit,
            "deductible": coverage.deductible
        }
        for peril, coverage in PRODUCTS[policy["product_code"]].items()
    ]

In [47]:
CLAIM_PROFILE: dict[str, Peril] = {
    "water_damage":      Peril(annual_frequency=0.030, loss_min=1_000, loss_max=15_000),
    "theft":             Peril(annual_frequency=0.015, loss_min=500, loss_max=10_000),
    "storm":             Peril(annual_frequency=0.020, loss_min=500, loss_max=20_000),
    "accidental_damage": Peril(annual_frequency=0.020, loss_min=200, loss_max=3_000),
    "fire":              Peril(annual_frequency=0.0015, loss_min=5_000, loss_max=300_000),
}

def build_claims(policy, coverages):
    claims = []
    window_end = min(d for d in (policy["end_date"], policy["cancelled_at"], TODAY) if d)

    for coverage in coverages:
        freq, lo, hi = CLAIM_PROFILE[coverage["peril"]]
        if random.random() >= freq:
            continue

        loss = round(random.triangular(lo, hi, lo), 2)
        payable = max(min(loss, coverage["limit_amount"]) - coverage["deductible"], 0)

        if payable == 0 or random.random() < 0.08:
            outcome, paid = "denied", 0
        elif loss > coverage["limit_amount"]:
            outcome, paid = "partial", payable
        else:
            outcome, paid = "paid", payable

        claims.append({
            "claim_id": f"CLM-{policy['policy_id'][4:]}-{coverage['peril'][:3].upper()}",
            "policy_id": policy["policy_id"],
            "loss_date": fake.date_between(policy["start_date"], window_end),
            "peril": coverage["peril"],
            "loss_amount": loss,
            "amount_paid": round(paid, 2),
            "outcome": outcome,
        })
    return claims

In [48]:
from pathlib import Path

N_CUSTOMERS = 1_000
DATA_DIR = Path("../db/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

customers = [random_customer(i) for i in range(1, N_CUSTOMERS + 1)]
policies  = [random_policy(i, c) for i, c in enumerate(customers, 1)]
coverages = [cov for p in policies for cov in build_coverages(p)]
claims    = [cl for p in policies
                for cl in build_claims(p, [c for c in coverages if c["policy_id"] == p["policy_id"]])]

def write_csv(name, rows):
    path = DATA_DIR / f"{name}.csv"
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"{path}: {len(rows)} rows")

write_csv("customers", customers)
write_csv("policies", policies)
write_csv("coverages", coverages)
write_csv("claims_history", claims)

../db/data/customers.csv: 1000 rows
../db/data/policies.csv: 1000 rows
../db/data/coverages.csv: 3998 rows
../db/data/claims_history.csv: 62 rows
